In [1]:
!pip install findspark

In [2]:
!pip install pyspark

In [3]:
import pyspark

In [4]:
from pyspark.sql import SparkSession

In [5]:
from pyspark.sql.types import StructField,StructType,IntegerType,StringType

In [6]:
from pyspark.sql.functions import *

In [7]:
spark = SparkSession.builder.appName("Bajaj-finserv").getOrCreate()

In [8]:
!pip install spacy

In [9]:
import spacy

In [10]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 29.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [11]:
nlp = spacy.load("en_core_web_sm")


In [12]:
excluded_words = spacy.lang.en.stop_words.STOP_WORDS

In [13]:
print(excluded_words)

{'full', 'which', '’d', 'we', '‘ll', 'nothing', 'seeming', 'never', 'afterwards', 'next', 'mine', 'out', 're', 'these', 'well', 'n‘t', 'each', 'or', 'ours', 'our', 'he', 'within', '‘m', 'anyway', 'indeed', 'others', 'everyone', 'hereupon', 'can', 'that', 'show', 'would', 'are', 'must', 'somehow', 'still', 'your', 'beyond', 'other', 'third', 'sometimes', 'mostly', 'should', 'n’t', 'neither', 'an', 'since', 'also', 'first', 'below', 'anything', 'ten', 'thence', 'done', 'already', 'along', 'everything', 'beforehand', 'per', 'this', 'and', 'here', 'from', 'hereby', 'into', 'please', 'am', 'over', 'become', 'nowhere', 'amount', 'hundred', 'hence', 'before', 'few', 'much', 'than', 'get', 'beside', 'yet', 'ourselves', 'while', 'make', 'ca', 'something', 'themselves', 'whereby', 'until', 'perhaps', 'put', 'whereafter', 'not', 'wherein', 'of', 'some', 'own', '‘d', 'a', 'however', 'again', 'empty', 'up', 'except', 'amongst', '’m', "'ll", 'whom', 'herein', 'any', 'due', 'alone', 'off', 'anyone', 

In [14]:
schema = StructType([ \
    StructField("Name",StringType(),True), \
    StructField("Rating",IntegerType(),True), \
    StructField("Review",StringType(),True), \
    StructField("Date",StringType(),True), \
    StructField("Useful",IntegerType(),True), \
    StructField("Answered",StringType(),True)
    ])

In [15]:
 main_df = spark.read.csv('/content/reviews.txt', schema=schema, sep=']')


In [16]:
main_df = main_df.withColumn("Useful",
                             when(col("Useful").isNull(), 0)
                             .otherwise(col("Useful")))

In [17]:
main_df.show(40)

+--------------------+------+--------------------+--------------+------+--------+
|                Name|Rating|              Review|          Date|Useful|Answered|
+--------------------+------+--------------------+--------------+------+--------+
|   prathamesh dabade|     5|I like how easily...| April 9, 2026|     1|     Yes|
|           raj kiran|     1|very slow app com...|March 22, 2026|     0|     Yes|
|   mahesh kumarsagar|     1|No update on stat...|April 10, 2026|    10|     Yes|
|       Yogesh Sharma|     1|seems like scammi...|March 24, 2026|     4|     Yes|
|    swadhan martha22|     1|I am extremely di...| April 2, 2026|     2|     Yes|
|      ANAND MAKRAIYA|     1|My bike installme...| April 3, 2026|     3|     Yes|
|     ankit Choudhary|     1|It is a very bad ...|March 28, 2026|     5|     Yes|
|        nguka assumi|     1|this app is a fra...| April 1, 2026|     2|     Yes|
|       Naveen Thakur|     1|I don't know how ...|March 22, 2026|     8|     Yes|
|               

In [18]:
main_df.createOrReplaceTempView("bajaj")

In [19]:
cleaned = spark.sql("SELECT Name,Review,Useful FROM bajaj;")

In [20]:
cleaned.show(truncate=False)

+--------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|Name                |Review                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [21]:
words = main_df.select(explode(split(col("Review")," ")).alias("word"))

In [22]:
words_finder = words.withColumn("words_lower",lower(col("word")))

In [23]:
words_finder = words_finder.groupBy("words_lower").agg(count("words_lower").alias("count")).orderBy(desc("count"))

In [24]:
filtered_words = words_finder.filter(~col("words_lower").isin(excluded_words))

In [25]:
additional_words = ["but", "can't", "iam", "i'm", "very", "this", "it", "it's", "like", "time",
                    "i", "a", "of", "in", "for", "that", "have", "or", "be", "me", "if", "as",
                    'so', "any", "even", "your", 'get', "am", "at", "do", "has","the","to","and","my","is","not","with","from","no","on","when","don't","will","been","an","only","then","also","by","you","are","they","are","was","can","all","use","doesn't","had","than","up",".","we","never","many","every","what","why","while","there","now","just","make","its","try","their","used","after","still"]


In [26]:
filtered_words = words_finder.filter(~col("words_lower").isin(additional_words))

In [27]:
filtered_words.show(60)

+-----------+-----+
|words_lower|count|
+-----------+-----+
|        app|   20|
|       card|   11|
|        emi|   10|
|   customer|    7|
|       app.|    6|
| completely|    5|
|       care|    5|
|     number|    5|
|      shows|    5|
|   cashback|    5|
|      which|    5|
|    because|    4|
|   service.|    4|
|     option|    4|
|   whenever|    4|
|       able|    4|
|       want|    4|
|      bajaj|    4|
|  complaint|    4|
|    payment|    4|
|       loan|    3|
|        how|    3|
|     please|    3|
|       emi.|    3|
|        one|    3|
|     mobile|    3|
|     unable|    3|
|     longer|    3|
|      close|    3|
|      times|    3|
|    without|    3|
|       view|    3|
|   payments|    3|
|        due|    3|
|        see|    3|
|    details|    3|
|       paid|    3|
|     amount|    3|
|       made|    3|
|  receiving|    3|
|     didn't|    3|
|      also,|    3|
|    service|    3|
|    working|    3|
|       most|    3|
|      fraud|    3|
|installment|    3|


In [28]:
mainde = spark.sql("SELECT * FROM bajaj WHERE Rating < 4 AND (Review LIKE '%worst%');")

In [29]:
mainde.select(count("Review")).show()

+-------------+
|count(Review)|
+-------------+
|            2|
+-------------+



### END



In [30]:
from transformers import pipeline

In [ ]:
pipe = pipeline("text-classification", model="siebert/sentiment-roberta-large-english")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("siebert/sentiment-roberta-large-english")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("siebert/sentiment-roberta-large-english")

In [ ]:
inputs = tokenizer("Hello, you are waste", return_tensors="pt")
outputs = model(**inputs)

prediction_logits = outputs.logits

In [ ]:
print(outputs)

In [ ]:
# Summary count
good_count = sum(1 for _, s, _, _ in results if s == "GOOD")
bad_count  = sum(1 for _, s, _, _ in results if s == "BAD")
print(f"\nTotal Reviews : {len(results)}")
print(f"  GOOD (Positive) : {good_count}")
print(f"  BAD  (Negative) : {bad_count}")


In [ ]:
# Collect reviews from Spark dataframe
reviews_data = main_df.select("Name", "Review").collect()

# Classify each review (truncate to 512 chars to stay within model token limit)
results = []
for row in reviews_data:
    name = row["Name"]
    review_text = row["Review"][:512] if row["Review"] else ""
    prediction = pipe(review_text)[0]
    label = prediction["label"]          # POSITIVE or NEGATIVE
    score = round(prediction["score"] * 100, 2)
    sentiment = "GOOD" if label == "POSITIVE" else "BAD"
    results.append((name, sentiment, score, review_text))

print(f"{'Name':<25} {'Sentiment':<10} {'Confidence':>12}  Review (truncated)")
print("-" * 90)
for name, sentiment, score, review in results:
    print(f"{name:<25} {sentiment:<10} {score:>10.2f}%  {review[:60]}...")


### Text Classification — Review Sentiment (Good / Bad)